# Train the toxic-comment detector (Google Colab)

This notebook retrains the model from the **original Jigsaw `train.csv`** and saves it in the
format the bot loads. Every step calls code from the repository (`toxic_detector/`), so what
you train here is exactly what the bot serves.

**The story, cell by cell**
1. Check the GPU → 2. get the code → 3. choose settings → 4. upload the data
5. load, clean and split → 6. build the model → 7. train → 8. test (per-label metrics)
9. save → 10. check that the saved model gives the same scores as the trained one
11. try a few sentences → 12. run the tests → 13. (optional) publish to Hugging Face

**Before you start**
- `Runtime → Change runtime type → GPU` (a T4 is enough).
- Download `train.csv` from the Kaggle *Jigsaw Toxic Comment Classification Challenge*
  (you must accept the competition rules). Keep it on your machine or Drive. **Never commit it.**
- Run with `QUICK_RUN = True` first: it trains on a small sample in a few minutes, just to prove
  every cell works. Its metrics are **not** reportable. Then set it to `False` for the real run.

## 1. Is there a GPU?

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - change the runtime type!")

## 2. Get the code
Clones the repository and installs what training needs. `BRANCH` is the branch to train from
(`main` once the rewrite is merged).

In [ ]:
BRANCH = "main"
REPO = "https://github.com/yoavyo0807/toxic-comment-detector.git"

import os
if not os.path.isdir("toxic-comment-detector"):
    !git clone --quiet --depth 1 --branch {BRANCH} {REPO}
%cd toxic-comment-detector
!pip install --quiet -r requirements-train.txt
!git log -1 --format="Training from commit %h (%cd)"

## 3. Settings
All defaults come from `toxic_detector/config.py`. Change values **here**, not in the code:
whatever you set is saved next to the model, so the README can quote the real values.

In [ ]:
from toxic_detector.config import TrainConfig

QUICK_RUN = True          # True: small sample, 1 epoch, to test the notebook. False: the real run.

cfg = TrainConfig()       # e.g. TrainConfig(epochs=3, learning_rate=3e-5)
if QUICK_RUN:
    cfg.epochs = 1
SAMPLE_ROWS = 5000 if QUICK_RUN else None
print(cfg.to_dict(), "| QUICK_RUN:", QUICK_RUN)

## 4. The data
Upload Kaggle's `train.csv` (about 70 MB; it can take a minute). It is stored in `data/`, which
Git ignores. If it is already there from an earlier upload in this session, nothing happens.

In [ ]:
from pathlib import Path
DATA = Path("data/train.csv")

if not DATA.exists():
    from google.colab import files
    DATA.parent.mkdir(exist_ok=True)
    uploaded = files.upload()                 # opens a file picker
    name = next(iter(uploaded))
    Path(name).rename(DATA)
print(DATA, f"{DATA.stat().st_size / 1e6:.1f} MB")

## 5. Load, clean, split
`load_jigsaw` builds our three labels from Jigsaw's six columns and applies the shared
`preprocess()`; `split` makes a seeded, stratified 80/10/10 split.
The table shows how rare each label is. Keep it next to any metric you report.

In [ ]:
import lightning as L
from toxic_detector.data import load_jigsaw, split, label_summary

L.seed_everything(cfg.seed)                   # same seed -> same sample, split and initial head
df = load_jigsaw(DATA)
if SAMPLE_ROWS:
    df = df.sample(min(SAMPLE_ROWS, len(df)), random_state=cfg.seed).reset_index(drop=True)
train_df, val_df, test_df = split(df, cfg)
summary = label_summary(all=df, train=train_df, val=val_df, test=test_df)
summary

## 6. Build the model and the batches
Pretrained DistilBERT + a new, randomly initialised linear head (`toxic_detector/model.py`).

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from toxic_detector.data import CommentDataset, Collator
from toxic_detector.model import ToxicClassifier
from toxic_detector.training import ToxicTrainer

tokenizer = AutoTokenizer.from_pretrained(cfg.base_model)
model = ToxicClassifier.from_base(cfg.base_model)
collate = Collator(tokenizer, cfg.max_length)

def loader(frame, shuffle):
    return DataLoader(CommentDataset(frame), batch_size=cfg.batch_size, shuffle=shuffle,
                      collate_fn=collate, num_workers=2)

train_dl, val_dl, test_dl = loader(train_df, True), loader(val_df, False), loader(test_df, False)
print(f"{len(train_df)} train / {len(val_df)} val / {len(test_df)} test rows, "
      f"{len(train_dl)} training batches per epoch")

## 7. Train
Mixed precision (`16-mixed`) on GPU makes training roughly twice as fast with almost no
effect on quality. Validation runs at the end of every epoch; the per-epoch numbers are
written to `lightning_logs/` (CSV).

In [ ]:
from lightning.pytorch.loggers import CSVLogger

module = ToxicTrainer(model, cfg)
trainer = L.Trainer(
    max_epochs=cfg.epochs,
    accelerator="auto",
    precision="16-mixed" if torch.cuda.is_available() else "32-true",
    logger=CSVLogger("lightning_logs"),
    enable_checkpointing=False,   # we save the final model ourselves, in the bot's format
    log_every_n_steps=50,
)
trainer.fit(module, train_dl, val_dl)

## 8. Test: per-label metrics
Run **once**, at the end. The test set must not influence any choice (settings, thresholds),
otherwise it stops being an honest estimate.

In [ ]:
import pandas as pd
from toxic_detector.config import LABELS

trainer.test(module, test_dl, verbose=False)
results = module.test_results
test_loss = trainer.callback_metrics["test/loss"].item()

table = pd.DataFrame({m: {l: results[f"test/{m}/{l}"] for l in [*LABELS, "macro"]}
                      for m in ("auroc", "precision", "recall", "f1")}).round(4)
print(f"test loss: {test_loss:.4f}   (thresholds: {model.thresholds})")
table

## 9. Save
One folder, `model/`: encoder, tokenizer, head (`.safetensors`) and `detector_config.json`,
which also records the settings, the data summary and the test metrics of **this** run.

In [ ]:
import json, platform, transformers

model.cpu()
record = {
    "train_config": cfg.to_dict(),
    "quick_run": QUICK_RUN,
    "data": {"source": "Jigsaw Toxic Comment Classification Challenge, train.csv",
             "label_summary": summary.to_dict(orient="index")},
    "test_metrics": {"loss": test_loss, **results},
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__, "lightning": L.__version__},
}
model.save("model", tokenizer, extra=record)
Path("metrics.json").write_text(json.dumps(record, indent=2))
!ls -lh model

## 10. Same model in training and in the bot?
Reload the saved folder exactly as the bot does (`Detector.load`) and compare its scores with
the in-memory model on test comments. They must match: this is the check that the old version
never had (brief, issue 2).

In [ ]:
from toxic_detector.predict import Detector

bot_view = Detector.load("model", device="cpu")
sample = test_df["text"].head(64).tolist()

with torch.inference_mode():
    model.eval()
    enc = tokenizer(sample, padding=True, truncation=True, max_length=cfg.max_length, return_tensors="pt")
    trained = torch.sigmoid(model(enc["input_ids"], enc["attention_mask"]))
served = torch.tensor([[s[l] for l in LABELS] for s in bot_view.scores_batch(sample)])

diff = (trained - served).abs().max().item()
print(f"largest difference: {diff:.2e}")
assert diff < 1e-4, "the saved model does NOT reproduce the trained one"
print("OK: the bot will serve exactly the model that was tested.")

## 11. Try it
Scores are probabilities; a label fires at its threshold (0.5 for now).
Treat this as a demo, not as evidence: the metrics in step 8 are the evidence.

In [ ]:
examples = [
    "Thanks for fixing the article, great work!",
    "I disagree with your edit, can we discuss it on the talk page?",
    "You are an idiot and nobody wants you here.",
    "Shut up or I will find you.",
]
for text in examples:
    scores = bot_view.scores(text)
    print(f"{str(bot_view.predict(text)):42} {' '.join(f'{l}={p:.2f}' for l, p in scores.items())}  <- {text}")

## 12. Run the tests, including the real-model check

In [ ]:
!MODEL_PATH=model python -m pytest -q

## 13. Keep the model
**Download** a zip (always do this; Colab deletes everything when the session ends):

In [ ]:
!zip -qr model.zip model metrics.json
from google.colab import files
files.download("model.zip")
files.download("metrics.json")

**Or publish to Hugging Face** (optional). Put a *write* token in Colab's **Secrets** panel
(the key icon on the left) under the name `HF_TOKEN`, and allow this notebook to read it.
The token is never typed into the notebook. The repo is created **private**; make it public
on the website when you are ready. Do not publish a `QUICK_RUN` model.

In [ ]:
PUSH_TO_HUB = False
HF_REPO_ID = "<your-hf-username>/toxic-comment-detector"

if PUSH_TO_HUB:
    assert not QUICK_RUN, "Do not publish a QUICK_RUN model."
    assert "<" not in HF_REPO_ID, "Set HF_REPO_ID to your own username/repo first."
    from google.colab import userdata
    from huggingface_hub import HfApi
    api = HfApi(token=userdata.get("HF_TOKEN"))
    api.create_repo(HF_REPO_ID, private=True, exist_ok=True)
    api.upload_folder(folder_path="model", repo_id=HF_REPO_ID,
                      commit_message="Train on Jigsaw train.csv (see detector_config.json)")
    print(f"Done: https://huggingface.co/{HF_REPO_ID}  ->  set MODEL_PATH={HF_REPO_ID} for the bot")